# Redrob Hackathon — Sandbox / Demo

This notebook is the **sandbox link** required by `submission_spec.md` Section 10.5.

It does three things, in order:
1. Clones the submission's GitHub repo (so it's running the *actual* committed code, not a copy).
2. Lets you upload a small candidate sample (≤100 candidates — `sample_candidates.json` from the
   hackathon bundle works directly).
3. Runs `rank.py` end-to-end on that sample and shows the resulting ranked CSV.

This does **not** need to handle the full 100K pool — that's checked separately at Stage 3 from
the GitHub repo directly. This notebook is just a fast, low-stakes way to verify the code runs
at all, with no GPU and no network calls during the ranking step itself (cloning the repo and
installing dependencies happens *before* the timed ranking step, same as in the real submission).


## Step 1 — Clone the repo and install dependencies

In [ ]:
# >>> EDIT THIS to your actual GitHub repo URL before running <<<
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git"

!rm -rf redrob_submission
!git clone $REPO_URL redrob_submission
%cd redrob_submission
!pip install -q -r requirements.txt


## Step 2 — Provide a small candidate sample

Two options:

**Option A (quickest):** the repo already includes `sample_candidates.json` (50 candidates from
the hackathon bundle) committed at the repo root. The cell below converts it to the `.jsonl`
format `rank.py` expects.

**Option B:** upload your own small sample (≤100 candidates) as a `.jsonl` file using the file
upload widget in Colab's left sidebar, then point `CANDIDATES_PATH` at it instead.


In [ ]:
import json

# Option A: convert the bundled sample_candidates.json (a JSON array) into
# the .jsonl format rank.py expects (one JSON object per line).
with open("sample_candidates.json") as f:
    sample = json.load(f)

with open("sample.jsonl", "w") as f:
    for c in sample:
        f.write(json.dumps(c) + "\n")

print(f"Wrote sample.jsonl with {len(sample)} candidates")

CANDIDATES_PATH = "sample.jsonl"   # change this if using Option B instead


## Step 3 — Run the ranking pipeline

Note: with only ~50 candidates, the top-100 selection logic will simply return everyone
available (sorted by score) since there are fewer than 100 candidates in this sample —
that's expected and fine for a sandbox smoke test. The real 100K-candidate run (verified
separately, see README.md) produces a true top-100 from the full pool.


In [ ]:
import time
t0 = time.time()

!python rank.py --candidates {CANDIDATES_PATH} --out sandbox_output.csv

print(f"\nWall-clock time: {time.time() - t0:.1f}s")


## Step 4 — Inspect the output

In [ ]:
import pandas as pd

out = pd.read_csv("sandbox_output.csv")
print(f"Rows: {len(out)}")
out.head(10)


In [ ]:
# Quick sanity checks (mirrors self_check.py, abbreviated for the sandbox sample size)
ranks = out["rank"].tolist()
assert sorted(ranks) == list(range(1, len(out) + 1)), "Ranks should be sequential starting at 1"
scores = out["score"].tolist()
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1)), "Scores should be non-increasing with rank"
assert out["candidate_id"].is_unique, "candidate_id should be unique"
print("All sandbox-scale checks passed.")


---
**For the full submission**: the actual `submission.csv` (top-100 from the real 100,000-candidate
pool) is committed in the GitHub repo at the root, produced by running:

```bash
python rank.py --candidates ./candidates.jsonl --out ./submission.csv
```

on the full dataset locally — see `README.md` in the repo for measured runtime and design notes.
